In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
lm_raw = pd.read_sql("SELECT * FROM raw_landmarks", engine)
print("shape:", lm_raw.shape)
print(lm_raw.columns.tolist())
lm_raw.head()

shape: (2500, 6)
['city', 'name', 'latitude', 'longitude', 'source_url', 'distance_from_center_m']


,city,name,latitude,longitude,source_url,distance_from_center_m
0,Amsterdam,Joods Historisch Museum,52.366944,4.904028,https://en.wikipedia.org/?curid=9088510,73.1
1,Amsterdam,Mozes en Aäronkerk,52.368056,4.903056,https://en.wikipedia.org/?curid=41470849,87.1
2,Amsterdam,Portuguese Synagogue (Amsterdam),52.367500,4.905400,https://en.wikipedia.org/?curid=8095914,89.0
3,Amsterdam,Markenplein,52.368600,4.905300,https://en.wikipedia.org/?curid=7159646,137.8
4,Amsterdam,Stopera,52.367222,4.901667,https://en.wikipedia.org/?curid=3491514,170.5


In [3]:
lm = lm_raw.copy()

for c in ["latitude", "longitude", "distance_from_center_m"]:
    lm[c] = pd.to_numeric(lm[c], errors="coerce")
lm["name"] = lm["name"].astype("string").str.strip()
lm["city"] = lm["city"].astype("string").str.strip()

n0 = len(lm)
lm = lm.dropna(subset=["name", "latitude", "longitude"])
print("rows without name/coordinates removed:", n0 - len(lm))

n0 = len(lm)
lm = lm.drop_duplicates(subset=["city", "name"])
print("duplicates (city, name) removed:", n0 - len(lm))

bad_dist = (lm["distance_from_center_m"] < 0) | (lm["distance_from_center_m"] > 10000)
print("distance outside 0-10000 m:", int(bad_dist.sum()))
lm = lm[~bad_dist]

print(lm.groupby("city").size())

rows without name/coordinates removed: 0
duplicates (city, name) removed: 0
distance outside 0-10000 m: 0
city
Amsterdam    566
Barcelona    517
Berlin       924
Columbus     493
dtype: int64


In [4]:
CANDIDATES = ["hotel", "company", "manufacturing", "government", "church", "school", "buurt"]

low = lm["name"].str.lower()
counts = pd.DataFrame({w: low.str.contains(w, na=False).groupby(lm["city"]).sum()
                       for w in CANDIDATES})
print(counts)

           hotel  company  manufacturing  government  church  school  buurt
city                                                                       
Amsterdam      7        0              0           1       2       0     20
Barcelona      7        2              0           0       2       0      0
Berlin        10        1              0           1      17       0      0
Columbus      11       15              4           2      25       0      0


In [5]:
EXTRA_EXCLUDE = ["hotel", "company", "manufacturing", "government"]        # الكلمات اللي حبّتي تستبعديها

if EXTRA_EXCLUDE:
    pat = "|".join(EXTRA_EXCLUDE)
    n0 = len(lm)
    lm = lm[~lm["name"] .str.lower().str.contains(pat, na=False)]
    print("removed by EXTRA_EXCLUDE:", n0 - len(lm))

landmarks_clean = lm.reset_index(drop=True)
landmarks_clean.groupby("city").sample(10, random_state=1)[["city", "name", "distance_from_center_m"]]

removed by EXTRA_EXCLUDE: 57


,city,name,distance_from_center_m
440,Amsterdam,VVA/Spartaan,4482.1
530,Amsterdam,Vuurtoreneiland,7485.5
346,Amsterdam,Westermoskee,2961.1
323,Amsterdam,Chassébuurt,2689.2
474,Amsterdam,World of Ajax,6642.4
360,Amsterdam,"Willemspark, Amsterdam",3176.6
29,Amsterdam,Ajax Experience,459.1
547,Amsterdam,Dijkgraafplein,8129.7
245,Amsterdam,West-Indisch Huis,1606.9
65,Amsterdam,"Binnenstad, Amsterdam",736.1


In [6]:
landmarks_clean.to_csv(OUT / "landmarks_clean.csv", index=False)
landmarks_clean.to_parquet(OUT / "landmarks_clean.parquet", index=False)
print("saved:", landmarks_clean.shape)

saved: (2443, 6)
